# 01 — Real MQuAKE-T data → machine translation → QLoRA SFT

**No fabricated relationships or QA samples.** This notebook downloads the original `MQuAKE-T.json` from the official Princeton MQuAKE repository (real temporal/Wikidata facts), selects documented **two-hop** chains, and translates **only the published questions** into German with `Helsinki-NLP/opus-mt-en-de`. Original answers, aliases, case IDs, and source SHA-256 remain traceable.

**Scientific cautions**: MQuAKE-T is a **knowledge-editing** benchmark, adapted here for exploratory unlearning. Temporal answers can be date-sensitive. Generated German questions require human translation QA. For forget targets, two-hop evaluation questions are entirely held out during SFT; direct target questions are learned in both languages; subsequent unlearning is English-only. A correct downstream two-hop answer is **not automatically literal retrieval of the forgotten intermediate fact**.

Hardware: Kaggle GPU T4 ×2, Internet ON. Output: `mquake_t_records.json`, `source_manifest.json`, `translation_audit.csv`, `full_sft_adapter/`, `retain_only_adapter/`, `sft_sanity.csv`.

Sources: https://github.com/princeton-nlp/MQuAKE (choose `MQuAKE-T`, **not** the counterfactual MQuAKE-CF).

In [ ]:
%pip install -q "transformers>=4.48,<5" "peft>=0.15,<0.19" "accelerate>=1.2,<2" "bitsandbytes>=0.45" "sentencepiece>=0.2" "safetensors>=0.5" "pandas>=2" "requests>=2.31" "wandb>=0.31,<1"

In [ ]:
"""Real MQuAKE-T loader and reproducible split construction.

No synthetic fact generation. All ground truth comes from the original dataset.
Translations are injected into build_payload from a separate translation model.
"""
import hashlib, json, random, re
from pathlib import Path

SOURCE_URL = "https://raw.githubusercontent.com/princeton-nlp/MQuAKE/main/datasets/MQuAKE-T.json"
SOURCE_PAGE = "https://github.com/princeton-nlp/MQuAKE"
SOURCE_NAME = "MQuAKE-T"
SEED = 20261026

def norm(x):
    return re.sub(r"\s+", " ", str(x or "").strip()).casefold()

def get_text(d, key):
    return str(d.get(key, "") or "").strip() if isinstance(d,dict) else ""

def load_source(path=None):
    """Downloads the official source; fails rather than fabricating fallback rows."""
    if path:
        src = Path(path).read_bytes()
    else:
        import requests
        r = requests.get(SOURCE_URL, timeout=120)
        r.raise_for_status()
        src = r.content
    source_sha256 = hashlib.sha256(src).hexdigest()
    raw = json.loads(src.decode('utf-8-sig'))
    if not isinstance(raw,list) or len(raw)<100:
        raise ValueError('MQuAKE-T source unavailable or unexpectedly small. STOP: no fake fallback.')
    return raw, source_sha256

def build_cases(raw, n_forget=24, n_retain=48, seed=SEED):
    """Take documented 2-hop original factual chains, never counterfactual new_* labels.

    Check unambiguous first hop, two-hop dependency and duplicate targeted
    predicates. Supports dataset-version variation with explicit rejects.
    """
    cases, reasons = [], {}
    def reject(msg): reasons[msg] = reasons.get(msg,0)+1
    for idx, rec in enumerate(raw):
        if not isinstance(rec,dict): reject('invalid_record'); continue
        steps = rec.get('single_hops', [])
        if not isinstance(steps,list) or len(steps)!=2:
            reject('not_two_hops'); continue
        qs = rec.get('questions', [])
        if not isinstance(qs,list) or not qs or not isinstance(qs[0],str):
            reject('missing_multihop_question'); continue
        s0,s1 = steps
        if not isinstance(s0,dict) or not isinstance(s1,dict):
            reject('invalid_hops');continue
        target_question, target_answer = get_text(s0,'question'),get_text(s0,'answer')
        support_question,support_answer = get_text(s1,'question'),get_text(s1,'answer')
        multi_question,multi_answer = qs[0].strip(),get_text(rec,'answer')
        if not all([target_question,target_answer,support_question,support_answer,multi_question,multi_answer]):
            reject('missing_values');continue
        # last step is required to agree with final answer, after aliases if provided
        last_aliases = {norm(x) for x in s1.get('answer_alias',[]) if isinstance(x,str)}
        multi_aliases = {norm(x) for x in rec.get('answer_alias',[]) if isinstance(x,str)}
        if norm(support_answer)!=norm(multi_answer) and not (
            norm(support_answer) in multi_aliases or norm(multi_answer) in last_aliases
        ):
            reject('chain_last_answer_mismatch');continue
        if norm(target_answer)==norm(multi_answer):
            reject('trivial_equal_answers');continue
        case_id = rec.get('case_id',idx)
        # We keep the original case id for traceability, not as a generated synthetic id.
        out = {
            'fact_id':f'mquake_t_{case_id}', 'source_index':idx,'source_case_id':case_id,
            'depth':2,
            'target_question_en':target_question,'target_answer':target_answer,
            'target_aliases':list(s0.get('answer_alias',[]) or []),
            'support_question_en':support_question,'support_answer':support_answer,
            'support_aliases':list(s1.get('answer_alias',[]) or []),
            'multi_question_en':multi_question,'multi_answer':multi_answer,
            'multi_aliases':list(rec.get('answer_alias',[]) or []),
            'source_edit_count':len(rec.get('requested_rewrite',[]) or []),
            'source_triples':((rec.get('orig') or {}).get('triples_labeled') or []),
        }
        cases.append(out)
    rng = random.Random(seed)
    rng.shuffle(cases)
    chosen = []
    used_direct_questions,used_full_chains,used_case_ids=set(),set(),set()
    for r in cases:
        direct=norm(r['target_question_en'])
        chain=norm(r['multi_question_en'])
        cid=str(r['source_case_id'])
        if cid in used_case_ids or direct in used_direct_questions or chain in used_full_chains:
            reject('duplicate_case_or_target');continue
        # Prevent same raw Wikidata first relation from landing in both splits.
        first_triple=(r.get('source_triples') or [[]])[0]
        edge=json.dumps(first_triple,sort_keys=True,ensure_ascii=False)
        if len(first_triple)==3 and edge in used_full_chains:
            reject('duplicate_first_fact');continue
        chosen.append(r)
        used_direct_questions.add(direct)
        used_full_chains.add(chain)
        if len(first_triple)==3: used_full_chains.add(edge)
        used_case_ids.add(cid)
        if len(chosen)==n_forget+n_retain: break
    if len(chosen)<n_forget+n_retain:
        raise RuntimeError(f'Only {len(chosen)} valid 2-hop unique factual chains, wanted {n_forget+n_retain}; exclusions: {reasons}. Reduce split sizes or inspect source; do not fabricate.')
    for i,r in enumerate(chosen):
        r['split']='forget' if i < n_forget else 'retain'
    return chosen,reasons

def get_translation_inputs(cases):
    result=[]
    for r in cases:
        for key in ('target_question_en','support_question_en','multi_question_en'):
            q=r[key]
            if q not in result:result.append(q)
    return result

def build_payload(cases,translations,source_sha256,rejections,model_id='Qwen/Qwen2.5-7B-Instruct'):
    """Build datasets using original published questions+answers and MT questions.
    Multi-hop target questions remain held out of SFT and of English forget loss.
    """
    tr_keys = ('target_question_en','support_question_en','multi_question_en')
    for r in cases:
        for key in tr_keys:
            q=r[key]
            if q not in translations or not str(translations[q]).strip():
                raise ValueError(f'Missing German translation of original benchmark question: {q}')
            if norm(translations[q])==norm(q):
                raise ValueError(f'Unchanged EN=>DE translation: {q}')
            r[key.replace('_en','_de')]=translations[q].strip()
    train=[]
    for r in cases:
        for lang in ('en','de'):
            train.append(dict(fact_id=r['fact_id'],split=r['split'],relation='target',lang=lang,
                              q=r[f'target_question_{lang}'],a=r['target_answer']))
            train.append(dict(fact_id=r['fact_id'],split='retain_support' if r['split']=='forget' else 'retain',relation='support',lang=lang,
                              q=r[f'support_question_{lang}'],a=r['support_answer']))
            if r['split']=='retain':
                train.append(dict(fact_id=r['fact_id'],split='retain',relation='multi',lang=lang,
                                  q=r[f'multi_question_{lang}'],a=r['multi_answer']))
    forget_train=[x for x in train if x['split']=='forget' and x['relation']=='target' and x['lang']=='en']
    retain_reference=[x for x in train if not (x['split']=='forget' and x['relation']=='target')]
    assert len(forget_train)==sum(r['split']=='forget' for r in cases)
    assert not any(x['split']=='forget' and x['relation']=='target' for x in retain_reference)
    assert not any(x['split']=='forget' and x['relation']=='multi' for x in train)
    assert all(not x['q'].startswith('example') for x in train)
    return dict(schema_version=2,dataset=SOURCE_NAME,source_url=SOURCE_URL,source_repo=SOURCE_PAGE,
                source_sha256=source_sha256,model_id=model_id,seed=SEED,
                translation_model='Helsinki-NLP/opus-mt-en-de',translation_scope='questions_only',
                source_rejection_counts=rejections,
                records=cases,train=train,forget_train=forget_train,retain_reference=retain_reference,
                scientific_caveat='MQuAKE-T evaluates real temporal facts. These are knowledge-editing chains, adapted to an unlearning pilot. Downstream answer recovery is not necessarily literal recovery of the forgotten first-hop entity. Retain-only retraining on a pretrained model is not a certified forgotten model.')

In [ ]:
import os,gc,json,random,re,time,pathlib,unicodedata,math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel

MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
SEED=20261026
MAX_SEQ=384
ROOT=pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle').exists() else './kaggle_output')
ROOT.mkdir(parents=True,exist_ok=True)
SYSTEM='Answer the factual question in English with only the short entity name or phrase. Do not explain. If unsure, say UNKNOWN.'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def require_gpu():
    assert torch.cuda.is_available(),'Enable a GPU accelerator in Kaggle settings.'
    print('GPU devices:',[(i,torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])

def cleanup():
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def get_tokenizer():
    t=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
    if t.pad_token is None:t.pad_token=t.eos_token
    return t

def get_backbone():
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=q,torch_dtype=torch.float16,device_map={'':0},attn_implementation='sdpa')

def make_qa(e,tok):
    prompt=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':e['q']}],tokenize=True,add_generation_prompt=True)
    if isinstance(prompt,torch.Tensor):prompt=prompt.tolist()
    ans=tok.encode(e['a'],add_special_tokens=False)+[tok.eos_token_id]
    if len(prompt)+len(ans)>MAX_SEQ:
        raise ValueError(f'Example exceeds MAX_SEQ={MAX_SEQ}, increase limit or filter: {e["q"][:60]}')
    ids=prompt+ans
    return {'input_ids':torch.tensor([ids],dtype=torch.long),
            'labels':torch.tensor([[-100]*len(prompt)+ans],dtype=torch.long),
            'attention_mask':torch.ones((1,len(ids)),dtype=torch.long)}

def to_gpu(b):return {k:v.to('cuda:0') for k,v in b.items()}

def seq_logp(model,b):
    outs=model(input_ids=b['input_ids'],attention_mask=b['attention_mask'])
    logits=outs.logits[:,:-1,:].float()
    labels=b['labels'][:,1:]
    mask=labels.ne(-100)
    selected=torch.log_softmax(logits,dim=-1).gather(-1,labels.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    return (selected*mask).sum(dim=-1)

def generate_answer(model,tok,q,max_new_tokens=30):
    model.eval()
    prefix=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':q}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda:0')
    with torch.inference_mode():
        out=model.generate(input_ids=prefix,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id,eos_token_id=tok.eos_token_id)
    return tok.decode(out[0,prefix.shape[-1]:],skip_special_tokens=True).strip()

def normalize_answer(text):
    t=unicodedata.normalize('NFKC',str(text or '')).casefold().strip()
    t=re.sub(r'[^\w\s]', ' ',t)
    return re.sub(r'\s+',' ',t).strip()

def match_answer(response,gold,aliases=()):
    """Conservative exact/short-contained matching. Review outputs manually for publication."""
    p=normalize_answer(response)
    if not p or 'unknown' in p or 'i don t know' in p:return 0
    choices=[normalize_answer(x) for x in [gold]+list(aliases or []) if str(x).strip()]
    choices=[x for x in choices if x and len(x)>1]
    if p in choices:return 1
    if len(p)>75:return 0
    return int(any(re.search(r'(?<!\w)'+re.escape(c)+r'(?!\w)',p) is not None for c in choices if len(c)>2))

In [ ]:
# Weights & Biases tracking for CoreWeave Forge.
# Kaggle secrets are separate from GitHub Actions secrets.
import os
import wandb

WANDB_ENTITY = "crosslingual"
WANDB_PROJECT = "multilingual_deepunlearning"
WANDB_DASHBOARD = "https://forge.coreweave.com/wandb/crosslingual/multilingual_deepunlearning"
WANDB_GROUP = "mquake-t-20261026"
# The Forge API endpoint is NOT the same URL as the website/dashboard.
os.environ.setdefault("WANDB_BASE_URL", "https://api.forge.coreweave.com")

def start_tracking(stage, name, config=None):
    token = os.environ.get("WANDB_API_KEY")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception as exc:
            raise RuntimeError(
                "W&B key unavailable. In EACH Kaggle notebook editor, open "
                "Add-ons > Secrets, add WANDB_API_KEY, and enable access for "
                "this notebook. GitHub Actions secrets are not passed to Kaggle."
            ) from exc
    if not token:
        raise RuntimeError("WANDB_API_KEY is empty; stop before starting GPU work.")
    os.environ["WANDB_API_KEY"] = token  # NEVER print this variable
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type=stage,
        name=name,
        config=config or {},
        mode="online",
        settings=wandb.Settings(init_timeout=120),
    )
    if run is None:
        raise RuntimeError("W&B did not start an online tracking run.")
    print("Tracking to:", WANDB_DASHBOARD, "run:", run.id)
    return run


In [ ]:
# Data and training configuration. Keep a tiny pilot first, then increase after competence check.
N_FORGET=24
N_RETAIN=48
EPOCHS=2
SFT_LR=2e-4
ACCUM=8
LORA_R=8
TRAIN_RETAIN_ONLY=True
ORIGINAL_MQUAKE_T_PATH=None  # optional exact path to official MQuAKE-T.json attached as Kaggle input
prep_run = start_tracking("data", "mquake-t-source-and-translation", {"seed": SEED, "forget_n": N_FORGET, "retain_n": N_RETAIN, "source": "MQuAKE-T", "translation_model": "Helsinki-NLP/opus-mt-en-de"})
require_gpu()
raw,source_sha=load_source(ORIGINAL_MQUAKE_T_PATH)
cases,rejections=build_cases(raw,N_FORGET,N_RETAIN,seed=SEED)
print('Source rows:',len(raw),'selected:',len(cases),'sha256:',source_sha)
print('Selection exclusions:',rejections)
print('First REAL source case:',cases[0]['source_case_id'],cases[0]['target_question_en'],cases[0]['multi_question_en'])
prep_run.log({"data/source_rows": len(raw), "data/selected_cases": len(cases), "data/forget_cases": N_FORGET, "data/retain_cases": N_RETAIN})
prep_run.config.update({"source_sha256": source_sha}, allow_val_change=True)

In [ ]:
# Translate only existing benchmark questions. No new entities, relations, or gold answers.
from transformers import MarianMTModel,MarianTokenizer
MT_MODEL='Helsinki-NLP/opus-mt-en-de'
mt_tok=MarianTokenizer.from_pretrained(MT_MODEL)
mt_model=MarianMTModel.from_pretrained(MT_MODEL).to('cuda:0').eval()
questions=get_translation_inputs(cases)
translations={}
for start in range(0,len(questions),12):
    batch=questions[start:start+12]
    enc=mt_tok(batch,padding=True,truncation=True,max_length=224,return_tensors='pt').to('cuda:0')
    with torch.inference_mode():
        generated=mt_model.generate(**enc,max_new_tokens=224,num_beams=4,do_sample=False)
    translations.update(zip(batch,mt_tok.batch_decode(generated,skip_special_tokens=True)))
    if start%60==0:print('Translated',min(start+len(batch),len(questions)),'/',len(questions))
del mt_model,mt_tok,enc,generated
cleanup()
audit=pd.DataFrame([{'source_question_en':a,'translated_question_de':b,'same_text':norm(a)==norm(b)} for a,b in translations.items()])
audit.to_csv(ROOT/'translation_audit.csv',index=False)
print(audit.head(12).to_string(index=False))
if audit.same_text.any():
    raise RuntimeError('Some machine translations were unchanged. Inspect translation_audit.csv; do not silently treat untranslated prompts as German.')
print('RESEARCH REQUIREMENT: Audit translated questions for meaning, names, and relation order before drawing conclusions.')
prep_run.log({"data/translation_count": len(translations), "data/unchanged_translations": int(audit.same_text.sum()), "data/translation_audit": wandb.Table(dataframe=audit)})
prep_run.finish()

In [ ]:
payload=build_payload(cases,translations,source_sha,rejections,model_id=MODEL_ID)
(ROOT/'mquake_t_records.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2))
(ROOT/'source_manifest.json').write_text(json.dumps({
    'original_dataset':'MQuAKE-T','source_url':SOURCE_URL,'source_sha256':source_sha,
    'original_rows':len(raw),'selected_rows':len(cases),'forget':N_FORGET,'retain':N_RETAIN,
    'selection_rejections':rejections,'translation_model':MT_MODEL,'seed':SEED,
    'original_case_ids':[r['source_case_id'] for r in cases],
    'warning':'questions machine translated: needs manual QA; not a new native multilingual benchmark'
},indent=2,ensure_ascii=False))
train=payload['train']
retain_reference=payload['retain_reference']
tokenizer=get_tokenizer()
max_len=max(make_qa(e,tokenizer)['input_ids'].shape[-1] for e in train)
print('SFT examples:',len(train),'retrain ref:',len(retain_reference),'max token length:',max_len)
assert all(r['source_case_id'] is not None for r in cases)
assert not any(x['relation']=='multi' and x['split']=='forget' for x in train)

In [ ]:
def train_adapter(rows,name,seed_shift=0):
    run = start_tracking("sft", name, {"seed": SEED+seed_shift, "examples": len(rows), "epochs": EPOCHS, "learning_rate": SFT_LR, "accumulation_steps": ACCUM, "lora_rank": LORA_R, "backbone": MODEL_ID, "dataset": "MQuAKE-T"})
    run.define_metric("train/optimizer_step")
    run.define_metric("train/*", step_metric="train/optimizer_step")
    optim_step = 0
    random.seed(SEED+seed_shift);torch.manual_seed(SEED+seed_shift)
    base=get_backbone()
    base.config.use_cache=False
    base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True)
    config=LoraConfig(r=LORA_R,lora_alpha=2*LORA_R,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model=get_peft_model(base,config)
    model.train()
    optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=SFT_LR)
    for epoch in range(EPOCHS):
        order=list(range(len(rows)));random.shuffle(order)
        optimizer.zero_grad(set_to_none=True)
        rolling=0
        for j,i in enumerate(order):
            b=to_gpu(make_qa(rows[i],tokenizer))
            loss=model(**b).loss
            (loss/ACCUM).backward()
            rolling+=float(loss.detach())
            if (j+1)%ACCUM==0 or j+1==len(order):
                torch.nn.utils.clip_grad_norm_(model.parameters(),1)
                optimizer.step();optimizer.zero_grad(set_to_none=True)
                optim_step += 1
                run.log({"train/optimizer_step": optim_step, "train/ce_loss": float(loss.detach()), "train/epoch_fraction": epoch+(j+1)/len(order), "train/lr": SFT_LR, "system/gpu_peak_gib": torch.cuda.max_memory_allocated(0)/1024**3})
            if (j+1)%100==0 or j+1==len(order):
                print(name,'epoch',epoch+1,'rows',j+1,'/',len(order),'mean_ce',round(rolling/(j+1),4))
    model.save_pretrained(ROOT/name,safe_serialization=True)
    tokenizer.save_pretrained(ROOT/name)
    print('Saved',ROOT/name)
    run.summary["train/final_ce_loss"] = float(loss.detach())
    run.summary["train/optimizer_steps"] = optim_step
    run.finish()
    return model

full=train_adapter(train,'full_sft_adapter')

In [ ]:
# Prerequisite evaluation on the SAME source IDs and direct/held-out 2-hop queries.
rows=[]
for r in ([x for x in cases if x['split']=='forget'][:8]+[x for x in cases if x['split']=='retain'][:8]):
    for lang in ('en','de'):
        for kind in ('direct','multi'):
            q=r[f'{"target" if kind=="direct" else "multi"}_question_{lang}']
            gold=r['target_answer'] if kind=='direct' else r['multi_answer']
            aliases=r['target_aliases'] if kind=='direct' else r['multi_aliases']
            pred=generate_answer(full,tokenizer,q)
            rows.append(dict(checkpoint='full_sft',case_id=r['source_case_id'],split=r['split'],language=lang,probe=kind,
                             question=q,gold=gold,prediction=pred,correct=match_answer(pred,gold,aliases)))
sanity_df = pd.DataFrame(rows)
sanity_df.to_csv(ROOT/'sft_sanity.csv',index=False)
sanity_by_slice = sanity_df.groupby(['split','language','probe'],as_index=False).agg(accuracy=('correct','mean'), count=('correct','size'))
print(sanity_by_slice.to_string(index=False))
sanity_run = start_tracking("evaluation", "sft-sanity-english-german", {"checkpoint": "full_sft", "num_probed_cases": 16, "source_sha256": source_sha})
sanity_metrics = {"eval/"+str(row.split)+"/"+str(row.language)+"/"+str(row.probe)+"_accuracy": float(row.accuracy) for row in sanity_by_slice.itertuples()}
sanity_metrics["eval/predictions"] = wandb.Table(dataframe=sanity_df)
sanity_metrics["eval/accuracy_by_slice"] = wandb.Table(dataframe=sanity_by_slice)
sanity_run.log(sanity_metrics)
sanity_run.finish()
print('STOP before unlearning if full SFT has poor direct or two-hop ability in either language.')
del full
cleanup()
if TRAIN_RETAIN_ONLY:
    retrain=train_adapter(retain_reference,'retain_only_adapter',seed_shift=1)
    del retrain
    cleanup()
print('Notebook 01 complete. Save Kaggle version with all outputs before Notebook 02.')

**Interpretation prerequisite:** Benchmark facts remain real temporal source facts, but data values can be outdated or ambiguous. Inspect `translation_audit.csv`, `sft_sanity.csv`, and representative benchmark source entries before proceeding. The retain-only adapter is an imperfect retraining reference because the pretrained backbone may already know the real-world target facts.